Importation des bibliothèques

In [4]:
# ============================================================
# ÉTAPE 0 — AUDIT DES 4 SOURCES DE DONNÉES
# Projet : Prédiction et évaluation de la performance académique
# Objectif : analyser la qualité des sources avant fusion
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)

Définition des chemins

In [13]:
# ============================================================
# 0.1. Chemins des fichiers
# ============================================================

DATA_DIR = Path("../data")

PATH_INSCRIPTIONS = DATA_DIR / "inscriptions.json"
PATH_TEST_INITIAL = DATA_DIR / "tests.json"
PATH_TESTS_INTERMEDIAIRES = DATA_DIR / "diego-mi-test.xlsx"
PATH_TESTS_FINAUX = DATA_DIR / "RESULTATS_FINAL_TESTS_PAR_PARCOURS.xlsx"

fichiers = {
    "Données d'inscription": PATH_INSCRIPTIONS,
    "Test initial Q1-Q20": PATH_TEST_INITIAL,
    "Tests intermédiaires": PATH_TESTS_INTERMEDIAIRES,
    "Tests finaux": PATH_TESTS_FINAUX
}

for nom, chemin in fichiers.items():
    print(f"{nom:30s} | existe = {chemin.exists()} | chemin = {chemin}")

Données d'inscription          | existe = True | chemin = ../data/inscriptions.json
Test initial Q1-Q20            | existe = True | chemin = ../data/tests.json
Tests intermédiaires           | existe = True | chemin = ../data/diego-mi-test.xlsx
Tests finaux                   | existe = True | chemin = ../data/RESULTATS_FINAL_TESTS_PAR_PARCOURS.xlsx


Cellule 3 — Chargement des données

In [14]:
# ============================================================
# 0.2. Chargement des quatre sources
# ============================================================

df_inscriptions = pd.read_json(PATH_INSCRIPTIONS)
df_test_initial = pd.read_json(PATH_TEST_INITIAL)

tests_intermediaires = pd.read_excel(PATH_TESTS_INTERMEDIAIRES, sheet_name=None)
tests_finaux = pd.read_excel(PATH_TESTS_FINAUX, sheet_name=None)

print("Chargement terminé.")
print("Données d'inscription :", df_inscriptions.shape)
print("Test initial :", df_test_initial.shape)
print("Feuilles des tests intermédiaires :", list(tests_intermediaires.keys()))
print("Feuilles des tests finaux :", list(tests_finaux.keys()))

Chargement terminé.
Données d'inscription : (472, 19)
Test initial : (315, 25)
Feuilles des tests intermédiaires : ['MI-TEST-DA', 'MI-TEST-BI', 'MI-TEST-DS', 'MI-TEST-IA']
Feuilles des tests finaux : ['FINAL-TEST-DA', 'FINAL-TEST-BI', 'FINAL-TEST-DS', 'FINAL-TEST-IA']


Cellule 4 — Fonction de normalisation des identifiants

In [15]:
# ============================================================
# 0.3. Fonction de normalisation des identifiants
# ============================================================

def normaliser_identifiant(serie):
    """
    Normalise les identifiants pour faciliter les contrôles :
    - conversion en texte ;
    - suppression des espaces avant/après ;
    - remplacement des valeurs vides par NaN.
    """
    return (
        serie.astype(str)
        .str.strip()
        .replace({
            "": np.nan,
            "nan": np.nan,
            "None": np.nan,
            "NaT": np.nan
        })
    )

Cellule 5 — Détection automatique de la colonne identifiant

In [16]:
# ============================================================
# 0.4. Détection automatique de la colonne identifiant
# ============================================================

def detecter_colonne_identifiant(df):
    """
    Recherche une colonne pouvant servir d'identifiant.
    La détection reste prudente : elle ne crée pas de colonne,
    elle signale seulement les colonnes candidates.
    """
    candidats = []
    
    for col in df.columns:
        col_min = str(col).lower()
        if "identification" in col_min or "identifiant" in col_min or col_min == "id":
            candidats.append(col)
    
    return candidats


print("Identifiants candidats — inscriptions :", detecter_colonne_identifiant(df_inscriptions))
print("Identifiants candidats — test initial :", detecter_colonne_identifiant(df_test_initial))

for nom_feuille, df in tests_intermediaires.items():
    print(f"Identifiants candidats — test intermédiaire {nom_feuille} :", detecter_colonne_identifiant(df))

for nom_feuille, df in tests_finaux.items():
    print(f"Identifiants candidats — test final {nom_feuille} :", detecter_colonne_identifiant(df))

Identifiants candidats — inscriptions : ['IDENTIFICATION']
Identifiants candidats — test initial : ['Identification']
Identifiants candidats — test intermédiaire MI-TEST-DA : ['IDENTIFICATION', 'Id']
Identifiants candidats — test intermédiaire MI-TEST-BI : ['IDENTIFICATION', 'Id']
Identifiants candidats — test intermédiaire MI-TEST-DS : ['IDENTIFICATION', 'Id']
Identifiants candidats — test intermédiaire MI-TEST-IA : ['IDENTIFICATION', 'Id']
Identifiants candidats — test final FINAL-TEST-DA : ['IDENTIFICATION']
Identifiants candidats — test final FINAL-TEST-BI : ['IDENTIFICATION']
Identifiants candidats — test final FINAL-TEST-DS : ['IDENTIFICATION']
Identifiants candidats — test final FINAL-TEST-IA : ['IDENTIFICATION']


Cellule 6 — Fonction d’audit général

In [17]:
# ============================================================
# 0.5. Fonction d'audit général d'un DataFrame
# ============================================================

def audit_dataframe(df, nom_source, colonne_id=None):
    """
    Produit un résumé général de qualité pour une source.
    """
    
    nb_lignes, nb_colonnes = df.shape
    total_cellules = nb_lignes * nb_colonnes
    
    rapport = {
        "source": nom_source,
        "nb_lignes": nb_lignes,
        "nb_colonnes": nb_colonnes,
        "nb_valeurs_manquantes": int(df.isna().sum().sum()),
        "taux_valeurs_manquantes_%": round((df.isna().sum().sum() / total_cellules) * 100, 2) if total_cellules > 0 else 0,
        "nb_lignes_dupliquees": int(df.astype(str).duplicated().sum())
    }
    
    if colonne_id is not None and colonne_id in df.columns:
        ids = normaliser_identifiant(df[colonne_id])
        rapport["colonne_identifiant"] = colonne_id
        rapport["identifiants_manquants"] = int(ids.isna().sum())
        rapport["identifiants_uniques"] = int(ids.dropna().nunique())
        rapport["doublons_identifiants"] = int(ids.dropna().duplicated().sum())
    else:
        rapport["colonne_identifiant"] = "Non détectée"
        rapport["identifiants_manquants"] = np.nan
        rapport["identifiants_uniques"] = np.nan
        rapport["doublons_identifiants"] = np.nan
    
    return rapport

Cellule 7 — Audit global des sources

In [19]:
# ============================================================
# 0.6. Audit global des 4 sources
# ============================================================

rapports = []

# À adapter seulement si les noms exacts diffèrent dans tes fichiers
COL_ID_INSCRIPTIONS = "IDENTIFICATION"
COL_ID_TEST_INITIAL = "Identification"
COL_ID_EXCEL = "IDENTIFICATION"

rapports.append(
    audit_dataframe(df_inscriptions, "Données d'inscription", COL_ID_INSCRIPTIONS)
)

rapports.append(
    audit_dataframe(df_test_initial, "Test initial Q1-Q20", COL_ID_TEST_INITIAL)
)

for nom_feuille, df in tests_intermediaires.items():
    rapports.append(
        audit_dataframe(df, f"Test intermédiaire - {nom_feuille}", COL_ID_EXCEL)
    )

for nom_feuille, df in tests_finaux.items():
    rapports.append(
        audit_dataframe(df, f"Test final - {nom_feuille}", COL_ID_EXCEL)
    )

df_audit_global = pd.DataFrame(rapports)
df_audit_global

,source,nb_lignes,nb_colonnes,nb_valeurs_manquantes,taux_valeurs_manquantes_%,nb_lignes_dupliquees,colonne_identifiant,identifiants_manquants,identifiants_uniques,doublons_identifiants
0,Données d'inscription,472,19,506,5.64,0,IDENTIFICATION,0,472,0
1,Test initial Q1-Q20,315,25,661,8.39,0,Identification,10,305,0
2,Test intermédiaire - MI-TEST-DA,37,24,256,28.83,0,IDENTIFICATION,5,32,0
3,Test intermédiaire - MI-TEST-BI,65,24,375,24.04,0,IDENTIFICATION,5,60,0
4,Test intermédiaire - MI-TEST-DS,46,24,261,23.64,0,IDENTIFICATION,2,44,0
5,Test intermédiaire - MI-TEST-IA,62,24,341,22.92,0,IDENTIFICATION,2,60,0
6,Test final - FINAL-TEST-DA,34,22,25,3.34,0,IDENTIFICATION,0,34,0
7,Test final - FINAL-TEST-BI,60,22,34,2.58,0,IDENTIFICATION,0,60,0
8,Test final - FINAL-TEST-DS,48,22,11,1.04,0,IDENTIFICATION,0,48,0
9,Test final - FINAL-TEST-IA,64,22,21,1.49,0,IDENTIFICATION,0,64,0


Cellule 8 — Liste des colonnes disponibles

In [20]:
# ============================================================
# 0.7. Colonnes disponibles dans chaque source
# ============================================================

print("\n===== COLONNES — DONNÉES D'INSCRIPTION =====")
print(list(df_inscriptions.columns))

print("\n===== COLONNES — TEST INITIAL =====")
print(list(df_test_initial.columns))

print("\n===== COLONNES — TESTS INTERMÉDIAIRES =====")
for nom_feuille, df in tests_intermediaires.items():
    print(f"\n--- Feuille : {nom_feuille} ---")
    print(list(df.columns))

print("\n===== COLONNES — TESTS FINAUX =====")
for nom_feuille, df in tests_finaux.items():
    print(f"\n--- Feuille : {nom_feuille} ---")
    print(list(df.columns))


===== COLONNES — DONNÉES D'INSCRIPTION =====
['CreatedAt', 'UpdatedAt', 'Title', 'IDENTIFICATION', 'Année de naissance', 'Genre', "Région d'origine", 'Filière', "Niveau d'étude", 'Statut actuel', 'Niveau informatique', 'Niveau Excel', 'Niveau Power BI', 'Niveau Python', 'Niveau IA', 'Motivation', 'Objectif pro', 'Disponibilité', 'Source info']

===== COLONNES — TEST INITIAL =====
['CreatedAt', 'UpdatedAt', 'Date du test', 'Q1 - Resume ventes Excel', 'Q2 - Import CSV Excel', 'Q3 - Modele donnees Excel', 'Q4 - Mauvaise pratique visu Excel', 'Q5 - 2e grande valeur Excel', 'Q6 - Mesure vs Colonne DAX', 'Q7 - CA annee precedente DAX', 'Q8 - Vue Modele Power BI', 'Q9 - Acces directeurs regionaux', 'Q10 - 12 commerciaux 3 indicateurs', 'Q11 - Bibliotheque CSV Python', 'Q12 - Overfitting Underfitting', 'Q13 - Segmentation 50000 clients', 'Q14 - Deployer modele Python API', 'Q15 - Valeurs manquantes 30pc', 'Q16 - Role system prompt LLM', 'Q17 - Assistant IA PDF financiers', 'Q18 - Role embeddi

Cellule 9 — Valeurs manquantes par colonne

In [21]:
# ============================================================
# 0.8. Analyse des valeurs manquantes par colonne
# ============================================================

def audit_valeurs_manquantes(df, nom_source):
    """
    Retourne les valeurs manquantes par colonne.
    """
    return pd.DataFrame({
        "source": nom_source,
        "colonne": df.columns,
        "nb_manquants": df.isna().sum().values,
        "taux_manquants_%": (df.isna().sum().values / len(df) * 100).round(2)
    }).sort_values("taux_manquants_%", ascending=False)


liste_vm = []

liste_vm.append(audit_valeurs_manquantes(df_inscriptions, "Données d'inscription"))
liste_vm.append(audit_valeurs_manquantes(df_test_initial, "Test initial Q1-Q20"))

for nom_feuille, df in tests_intermediaires.items():
    liste_vm.append(audit_valeurs_manquantes(df, f"Test intermédiaire - {nom_feuille}"))

for nom_feuille, df in tests_finaux.items():
    liste_vm.append(audit_valeurs_manquantes(df, f"Test final - {nom_feuille}"))

df_valeurs_manquantes = pd.concat(liste_vm, ignore_index=True)

df_valeurs_manquantes.head(50)

,source,colonne,nb_manquants,taux_manquants_%
0,Données d'inscription,UpdatedAt,472,100.00
1,Données d'inscription,Source info,19,4.03
2,Données d'inscription,Région d'origine,13,2.75
3,Données d'inscription,Objectif pro,1,0.21
4,Données d'inscription,Genre,1,0.21
5,Données d'inscription,Niveau Excel,0,0.00
6,Données d'inscription,Disponibilité,0,0.00
7,Données d'inscription,Motivation,0,0.00
8,Données d'inscription,Niveau IA,0,0.00
9,Données d'inscription,Niveau Python,0,0.00


Cellule 10 — Audit des identifiants

In [22]:
# ============================================================
# 0.9. Audit des identifiants
# ============================================================

ids_inscriptions = normaliser_identifiant(df_inscriptions[COL_ID_INSCRIPTIONS])
ids_test_initial = normaliser_identifiant(df_test_initial[COL_ID_TEST_INITIAL])

ids_intermediaires_par_feuille = {}
for nom_feuille, df in tests_intermediaires.items():
    if COL_ID_EXCEL in df.columns:
        ids_intermediaires_par_feuille[nom_feuille] = normaliser_identifiant(df[COL_ID_EXCEL])
    else:
        ids_intermediaires_par_feuille[nom_feuille] = pd.Series(dtype="object")

ids_finaux_par_feuille = {}
for nom_feuille, df in tests_finaux.items():
    if COL_ID_EXCEL in df.columns:
        ids_finaux_par_feuille[nom_feuille] = normaliser_identifiant(df[COL_ID_EXCEL])
    else:
        ids_finaux_par_feuille[nom_feuille] = pd.Series(dtype="object")


resume_identifiants = []

resume_identifiants.append({
    "source": "Données d'inscription",
    "nb_identifiants_non_vides": ids_inscriptions.dropna().shape[0],
    "nb_identifiants_uniques": ids_inscriptions.dropna().nunique(),
    "nb_identifiants_dupliques": ids_inscriptions.dropna().duplicated().sum(),
    "nb_identifiants_manquants": ids_inscriptions.isna().sum()
})

resume_identifiants.append({
    "source": "Test initial Q1-Q20",
    "nb_identifiants_non_vides": ids_test_initial.dropna().shape[0],
    "nb_identifiants_uniques": ids_test_initial.dropna().nunique(),
    "nb_identifiants_dupliques": ids_test_initial.dropna().duplicated().sum(),
    "nb_identifiants_manquants": ids_test_initial.isna().sum()
})

for nom_feuille, ids in ids_intermediaires_par_feuille.items():
    resume_identifiants.append({
        "source": f"Test intermédiaire - {nom_feuille}",
        "nb_identifiants_non_vides": ids.dropna().shape[0],
        "nb_identifiants_uniques": ids.dropna().nunique(),
        "nb_identifiants_dupliques": ids.dropna().duplicated().sum(),
        "nb_identifiants_manquants": ids.isna().sum()
    })

for nom_feuille, ids in ids_finaux_par_feuille.items():
    resume_identifiants.append({
        "source": f"Test final - {nom_feuille}",
        "nb_identifiants_non_vides": ids.dropna().shape[0],
        "nb_identifiants_uniques": ids.dropna().nunique(),
        "nb_identifiants_dupliques": ids.dropna().duplicated().sum(),
        "nb_identifiants_manquants": ids.isna().sum()
    })

df_audit_identifiants = pd.DataFrame(resume_identifiants)
df_audit_identifiants

,source,nb_identifiants_non_vides,nb_identifiants_uniques,nb_identifiants_dupliques,nb_identifiants_manquants
0,Données d'inscription,472,472,0,0
1,Test initial Q1-Q20,305,305,0,10
2,Test intermédiaire - MI-TEST-DA,32,32,0,5
3,Test intermédiaire - MI-TEST-BI,60,60,0,5
4,Test intermédiaire - MI-TEST-DS,44,44,0,2
5,Test intermédiaire - MI-TEST-IA,60,60,0,2
6,Test final - FINAL-TEST-DA,34,34,0,0
7,Test final - FINAL-TEST-BI,60,60,0,0
8,Test final - FINAL-TEST-DS,48,48,0,0
9,Test final - FINAL-TEST-IA,64,64,0,0


Cellule 11 — Couverture entre les sources

In [23]:
# ============================================================
# 0.10. Couverture entre les 4 sources
# ============================================================

set_inscriptions = set(ids_inscriptions.dropna())
set_test_initial = set(ids_test_initial.dropna())

sets_intermediaires = [
    set(ids.dropna()) for ids in ids_intermediaires_par_feuille.values()
]

sets_finaux = [
    set(ids.dropna()) for ids in ids_finaux_par_feuille.values()
]

set_intermediaires_total = set().union(*sets_intermediaires) if len(sets_intermediaires) > 0 else set()
set_finaux_total = set().union(*sets_finaux) if len(sets_finaux) > 0 else set()

couverture = {
    "identifiants_inscriptions": len(set_inscriptions),
    "identifiants_test_initial": len(set_test_initial),
    "identifiants_tests_intermediaires_total": len(set_intermediaires_total),
    "identifiants_tests_finaux_total": len(set_finaux_total),
    "inscriptions_et_test_initial": len(set_inscriptions & set_test_initial),
    "inscriptions_et_tests_intermediaires": len(set_inscriptions & set_intermediaires_total),
    "inscriptions_et_tests_finaux": len(set_inscriptions & set_finaux_total),
    "test_initial_et_tests_intermediaires": len(set_test_initial & set_intermediaires_total),
    "test_initial_et_tests_finaux": len(set_test_initial & set_finaux_total),
    "inscriptions_test_initial_intermediaires_finaux": len(
        set_inscriptions & set_test_initial & set_intermediaires_total & set_finaux_total
    )
}

df_couverture = pd.DataFrame(
    list(couverture.items()),
    columns=["indicateur", "valeur"]
)

df_couverture

,indicateur,valeur
0,identifiants_inscriptions,472
1,identifiants_test_initial,305
2,identifiants_tests_intermediaires_total,196
3,identifiants_tests_finaux_total,206
4,inscriptions_et_test_initial,305
5,inscriptions_et_tests_intermediaires,141
6,inscriptions_et_tests_finaux,146
7,test_initial_et_tests_intermediaires,125
8,test_initial_et_tests_finaux,128
9,inscriptions_test_initial_intermediaires_finaux,125


Cellule 12 — Questions Q1 à Q20 du test initial

In [24]:
# ============================================================
# 0.11. Contrôle des questions Q1 à Q20
# ============================================================

colonnes_questions = [col for col in df_test_initial.columns if str(col).strip().upper().startswith("Q")]

print("Colonnes de questions détectées :")
print(colonnes_questions)
print("Nombre de questions détectées :", len(colonnes_questions))

questions_attendues = [f"Q{i}" for i in range(1, 21)]

questions_presentes = []
questions_absentes = []

for q in questions_attendues:
    trouvee = any(str(col).strip().upper().startswith(q.upper()) for col in df_test_initial.columns)
    if trouvee:
        questions_presentes.append(q)
    else:
        questions_absentes.append(q)

print("Questions attendues présentes :", questions_presentes)
print("Questions attendues absentes :", questions_absentes)

Colonnes de questions détectées :
['Q1 - Resume ventes Excel', 'Q2 - Import CSV Excel', 'Q3 - Modele donnees Excel', 'Q4 - Mauvaise pratique visu Excel', 'Q5 - 2e grande valeur Excel', 'Q6 - Mesure vs Colonne DAX', 'Q7 - CA annee precedente DAX', 'Q8 - Vue Modele Power BI', 'Q9 - Acces directeurs regionaux', 'Q10 - 12 commerciaux 3 indicateurs', 'Q11 - Bibliotheque CSV Python', 'Q12 - Overfitting Underfitting', 'Q13 - Segmentation 50000 clients', 'Q14 - Deployer modele Python API', 'Q15 - Valeurs manquantes 30pc', 'Q16 - Role system prompt LLM', 'Q17 - Assistant IA PDF financiers', 'Q18 - Role embedding dans RAG', 'Q19 - Agent IA selection outil', 'Q20 - Sortie fiable LLM tableau']
Nombre de questions détectées : 20
Questions attendues présentes : ['Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7', 'Q8', 'Q9', 'Q10', 'Q11', 'Q12', 'Q13', 'Q14', 'Q15', 'Q16', 'Q17', 'Q18', 'Q19', 'Q20']
Questions attendues absentes : []


Cellule 13 — Répartition Q1–Q20 par domaine

In [25]:
# ============================================================
# 0.12. Répartition théorique des questions par domaine
# ============================================================

repartition_questions = {
    "Excel / Analyse de données de base": [f"Q{i}" for i in range(1, 6)],
    "Business Intelligence / Power BI": [f"Q{i}" for i in range(6, 11)],
    "Data Science / Python / Machine Learning": [f"Q{i}" for i in range(11, 16)],
    "Intelligence Artificielle / LLM / RAG / agents IA": [f"Q{i}" for i in range(16, 21)]
}

df_repartition_questions = pd.DataFrame([
    {
        "domaine": domaine,
        "questions": ", ".join(questions),
        "nombre_questions": len(questions)
    }
    for domaine, questions in repartition_questions.items()
])

df_repartition_questions

,domaine,questions,nombre_questions
0,Excel / Analyse de données de base,"Q1, Q2, Q3, Q4, Q5",5
1,Business Intelligence / Power BI,"Q6, Q7, Q8, Q9, Q10",5
2,Data Science / Python / Machine Learning,"Q11, Q12, Q13, Q14, Q15",5
3,Intelligence Artificielle / LLM / RAG / agents IA,"Q16, Q17, Q18, Q19, Q20",5


Cellule 14 — Analyse rapide des réponses inconnues

In [26]:
# ============================================================
# 0.13. Analyse des réponses inconnues dans le test initial
# ============================================================

df_test_initial_audit = df_test_initial.copy()

colonnes_q_detectees = colonnes_questions

def compter_reponses_inconnues(row):
    """
    Compte les réponses qui semblent correspondre à une absence de réponse
    ou à une réponse de type 'je ne sais pas'.
    """
    valeurs = row.astype(str).str.lower()
    
    motifs = [
        "je ne sais pas",
        "ne sais pas",
        "aucune réponse",
        "sans réponse",
        "nan",
        "none"
    ]
    
    total = 0
    for val in valeurs:
        if any(motif in val for motif in motifs):
            total += 1
    return total

if len(colonnes_q_detectees) > 0:
    df_test_initial_audit["unknown_total_initial_audit"] = df_test_initial_audit[colonnes_q_detectees].apply(
        compter_reponses_inconnues,
        axis=1
    )
    
    print(df_test_initial_audit["unknown_total_initial_audit"].describe())
else:
    print("Aucune colonne Q détectée.")

count    315.000000
mean       8.571429
std        6.787761
min        0.000000
25%        2.500000
50%        7.000000
75%       15.000000
max       20.000000
Name: unknown_total_initial_audit, dtype: float64


Cellule 15 — Audit des années de naissance

In [27]:
# ============================================================
# 0.14. Audit des années de naissance
# ============================================================

colonnes_annee = [
    col for col in df_inscriptions.columns
    if "naissance" in str(col).lower() or "année" in str(col).lower() or "annee" in str(col).lower()
]

print("Colonnes candidates pour l'année de naissance :", colonnes_annee)

if len(colonnes_annee) > 0:
    col_annee = colonnes_annee[0]
    
    df_annee_audit = df_inscriptions[[COL_ID_INSCRIPTIONS, col_annee]].copy()
    df_annee_audit["annee_naissance_numeric"] = pd.to_numeric(
        df_annee_audit[col_annee],
        errors="coerce"
    )
    
    df_annees_suspectes = df_annee_audit[
        (df_annee_audit["annee_naissance_numeric"].isna()) |
        (df_annee_audit["annee_naissance_numeric"] < 1950) |
        (df_annee_audit["annee_naissance_numeric"] > 2015)
    ]
    
    print("Nombre d'années de naissance suspectes :", len(df_annees_suspectes))
    display(df_annees_suspectes.head(50))
else:
    df_annees_suspectes = pd.DataFrame()
    print("Aucune colonne d'année de naissance détectée.")

Colonnes candidates pour l'année de naissance : ['Année de naissance']
Nombre d'années de naissance suspectes : 9


,IDENTIFICATION,Année de naissance,annee_naissance_numeric
1,DIEG2026-002,20002,20002
5,DIEG2026-006,20002000,20002000
18,DIEG2026-019,2020055555200,2020055555200
26,DIEG2026-027,200001022006,200001022006
31,DIEG2026-032,200000,200000
33,DIEG2026-034,8032002,8032002
102,DIEG2026-103,20002003,20002003
198,DIEG2026-199,21,21
395,DIEG2026-396,24,24


Cellule 16 — Synthèse des feuilles Excel

In [28]:
# ============================================================
# 0.15. Synthèse des feuilles Excel
# ============================================================

synthese_feuilles = []

for nom_feuille, df in tests_intermediaires.items():
    synthese_feuilles.append({
        "type_test": "intermédiaire",
        "feuille": nom_feuille,
        "nb_lignes": df.shape[0],
        "nb_colonnes": df.shape[1],
        "colonnes": ", ".join(map(str, df.columns))
    })

for nom_feuille, df in tests_finaux.items():
    synthese_feuilles.append({
        "type_test": "final",
        "feuille": nom_feuille,
        "nb_lignes": df.shape[0],
        "nb_colonnes": df.shape[1],
        "colonnes": ", ".join(map(str, df.columns))
    })

df_synthese_feuilles = pd.DataFrame(synthese_feuilles)
df_synthese_feuilles

,type_test,feuille,nb_lignes,nb_colonnes,colonnes
0,intermédiaire,MI-TEST-DA,37,24,"IDENTIFICATION, Id, CreatedAt, UpdatedAt, nc_c..."
1,intermédiaire,MI-TEST-BI,65,24,"IDENTIFICATION, Id, CreatedAt, UpdatedAt, nc_c..."
2,intermédiaire,MI-TEST-DS,46,24,"IDENTIFICATION, Id, CreatedAt, UpdatedAt, nc_c..."
3,intermédiaire,MI-TEST-IA,62,24,"IDENTIFICATION, Id, CreatedAt, UpdatedAt, nc_c..."
4,final,FINAL-TEST-DA,34,22,"RowId, CreatedAt, UpdatedAt, InscriptionId, ID..."
5,final,FINAL-TEST-BI,60,22,"RowId, CreatedAt, UpdatedAt, InscriptionId, ID..."
6,final,FINAL-TEST-DS,48,22,"RowId, CreatedAt, UpdatedAt, InscriptionId, ID..."
7,final,FINAL-TEST-IA,64,22,"RowId, CreatedAt, UpdatedAt, InscriptionId, ID..."


Cellule 17 — Audit des doublons d’identifiants

In [29]:
# ============================================================
# 0.16. Détail des identifiants dupliqués
# ============================================================

def extraire_doublons_identifiants(df, colonne_id, nom_source):
    if colonne_id not in df.columns:
        return pd.DataFrame()
    
    ids = normaliser_identifiant(df[colonne_id])
    temp = df.copy()
    temp["_ID_NORMALISE"] = ids
    
    doublons = temp[
        temp["_ID_NORMALISE"].notna() &
        temp["_ID_NORMALISE"].duplicated(keep=False)
    ].copy()
    
    doublons["_SOURCE"] = nom_source
    return doublons

liste_doublons = []

liste_doublons.append(
    extraire_doublons_identifiants(df_inscriptions, COL_ID_INSCRIPTIONS, "Données d'inscription")
)

liste_doublons.append(
    extraire_doublons_identifiants(df_test_initial, COL_ID_TEST_INITIAL, "Test initial Q1-Q20")
)

for nom_feuille, df in tests_intermediaires.items():
    liste_doublons.append(
        extraire_doublons_identifiants(df, COL_ID_EXCEL, f"Test intermédiaire - {nom_feuille}")
    )

for nom_feuille, df in tests_finaux.items():
    liste_doublons.append(
        extraire_doublons_identifiants(df, COL_ID_EXCEL, f"Test final - {nom_feuille}")
    )

df_doublons_identifiants = pd.concat(
    [d for d in liste_doublons if not d.empty],
    ignore_index=True
) if any(not d.empty for d in liste_doublons) else pd.DataFrame()

df_doublons_identifiants.head(50)

""


Cellule 18 — Export du rapport d’audit

In [30]:
# ============================================================
# 0.17. Export des rapports d'audit
# ============================================================

OUTPUT_DIR = Path("outputs/00_audit_sources")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df_audit_global.to_csv(OUTPUT_DIR / "audit_global_sources.csv", index=False, encoding="utf-8-sig")
df_valeurs_manquantes.to_csv(OUTPUT_DIR / "audit_valeurs_manquantes.csv", index=False, encoding="utf-8-sig")
df_audit_identifiants.to_csv(OUTPUT_DIR / "audit_identifiants.csv", index=False, encoding="utf-8-sig")
df_couverture.to_csv(OUTPUT_DIR / "audit_couverture_sources.csv", index=False, encoding="utf-8-sig")
df_repartition_questions.to_csv(OUTPUT_DIR / "audit_repartition_questions.csv", index=False, encoding="utf-8-sig")
df_synthese_feuilles.to_csv(OUTPUT_DIR / "audit_feuilles_excel.csv", index=False, encoding="utf-8-sig")

if not df_annees_suspectes.empty:
    df_annees_suspectes.to_csv(OUTPUT_DIR / "audit_annees_naissance_suspectes.csv", index=False, encoding="utf-8-sig")

if not df_doublons_identifiants.empty:
    df_doublons_identifiants.to_csv(OUTPUT_DIR / "audit_doublons_identifiants.csv", index=False, encoding="utf-8-sig")

print("Rapports exportés dans :", OUTPUT_DIR)
print("Fichiers générés :")
for fichier in OUTPUT_DIR.iterdir():
    print("-", fichier.name)

Rapports exportés dans : outputs/00_audit_sources
Fichiers générés :
- audit_couverture_sources.csv
- audit_identifiants.csv
- audit_repartition_questions.csv
- audit_global_sources.csv
- audit_valeurs_manquantes.csv
- audit_annees_naissance_suspectes.csv
- audit_feuilles_excel.csv


Cellule 19 — Conclusion automatique de l’audit

In [31]:
# ============================================================
# 0.18. Conclusion automatique de l'audit
# ============================================================

print("===== CONCLUSION PROVISOIRE DE L'AUDIT =====")

print("\n1. Dimensions des sources")
display(df_audit_global)

print("\n2. Couverture des identifiants")
display(df_couverture)

print("\n3. Synthèse des feuilles Excel")
display(df_synthese_feuilles)

print("\n4. Questions du test initial")
display(df_repartition_questions)

print("\n5. Points à vérifier avant fusion")
print("- Harmoniser les colonnes d'identification.")
print("- Vérifier les identifiants manquants.")
print("- Vérifier les doublons d'identifiants.")
print("- Corriger ou isoler les années de naissance aberrantes.")
print("- Confirmer les feuilles DA, BI, DS et IA dans les tests intermédiaires et finaux.")
print("- Ne pas construire le dataset final tant que les clés de fusion ne sont pas validées.")

===== CONCLUSION PROVISOIRE DE L'AUDIT =====

1. Dimensions des sources


,source,nb_lignes,nb_colonnes,nb_valeurs_manquantes,taux_valeurs_manquantes_%,nb_lignes_dupliquees,colonne_identifiant,identifiants_manquants,identifiants_uniques,doublons_identifiants
0,Données d'inscription,472,19,506,5.64,0,IDENTIFICATION,0,472,0
1,Test initial Q1-Q20,315,25,661,8.39,0,Identification,10,305,0
2,Test intermédiaire - MI-TEST-DA,37,24,256,28.83,0,IDENTIFICATION,5,32,0
3,Test intermédiaire - MI-TEST-BI,65,24,375,24.04,0,IDENTIFICATION,5,60,0
4,Test intermédiaire - MI-TEST-DS,46,24,261,23.64,0,IDENTIFICATION,2,44,0
5,Test intermédiaire - MI-TEST-IA,62,24,341,22.92,0,IDENTIFICATION,2,60,0
6,Test final - FINAL-TEST-DA,34,22,25,3.34,0,IDENTIFICATION,0,34,0
7,Test final - FINAL-TEST-BI,60,22,34,2.58,0,IDENTIFICATION,0,60,0
8,Test final - FINAL-TEST-DS,48,22,11,1.04,0,IDENTIFICATION,0,48,0
9,Test final - FINAL-TEST-IA,64,22,21,1.49,0,IDENTIFICATION,0,64,0



2. Couverture des identifiants


,indicateur,valeur
0,identifiants_inscriptions,472
1,identifiants_test_initial,305
2,identifiants_tests_intermediaires_total,196
3,identifiants_tests_finaux_total,206
4,inscriptions_et_test_initial,305
5,inscriptions_et_tests_intermediaires,141
6,inscriptions_et_tests_finaux,146
7,test_initial_et_tests_intermediaires,125
8,test_initial_et_tests_finaux,128
9,inscriptions_test_initial_intermediaires_finaux,125



3. Synthèse des feuilles Excel


,type_test,feuille,nb_lignes,nb_colonnes,colonnes
0,intermédiaire,MI-TEST-DA,37,24,"IDENTIFICATION, Id, CreatedAt, UpdatedAt, nc_c..."
1,intermédiaire,MI-TEST-BI,65,24,"IDENTIFICATION, Id, CreatedAt, UpdatedAt, nc_c..."
2,intermédiaire,MI-TEST-DS,46,24,"IDENTIFICATION, Id, CreatedAt, UpdatedAt, nc_c..."
3,intermédiaire,MI-TEST-IA,62,24,"IDENTIFICATION, Id, CreatedAt, UpdatedAt, nc_c..."
4,final,FINAL-TEST-DA,34,22,"RowId, CreatedAt, UpdatedAt, InscriptionId, ID..."
5,final,FINAL-TEST-BI,60,22,"RowId, CreatedAt, UpdatedAt, InscriptionId, ID..."
6,final,FINAL-TEST-DS,48,22,"RowId, CreatedAt, UpdatedAt, InscriptionId, ID..."
7,final,FINAL-TEST-IA,64,22,"RowId, CreatedAt, UpdatedAt, InscriptionId, ID..."



4. Questions du test initial


,domaine,questions,nombre_questions
0,Excel / Analyse de données de base,"Q1, Q2, Q3, Q4, Q5",5
1,Business Intelligence / Power BI,"Q6, Q7, Q8, Q9, Q10",5
2,Data Science / Python / Machine Learning,"Q11, Q12, Q13, Q14, Q15",5
3,Intelligence Artificielle / LLM / RAG / agents IA,"Q16, Q17, Q18, Q19, Q20",5



5. Points à vérifier avant fusion
- Harmoniser les colonnes d'identification.
- Vérifier les identifiants manquants.
- Vérifier les doublons d'identifiants.
- Corriger ou isoler les années de naissance aberrantes.
- Confirmer les feuilles DA, BI, DS et IA dans les tests intermédiaires et finaux.
- Ne pas construire le dataset final tant que les clés de fusion ne sont pas validées.
